# DMET-SSTKG — Source S-STKG Component Ablation

This notebook reproduces the component-level S-STKG sensitivity analysis reported in **Table 4** of the manuscript.

The retained source phase-classifier checkpoint is kept fixed while one representation component is altered at a time on the held-out source-test split. The evaluated components are node geometry, motion, functional–anatomical role semantics, progress, the virtual-context node, edge descriptors, and temporal sampling.

The temporal-sampling condition requires the reconstruction-route pre-role S-STKG inputs because motion is recomputed on the sparser sampling grid. Those provider-derived reconstruction inputs are not redistributed with the released S-STKG archive.

Outputs are written under `outputs/validation_analyses/source_sstkg_component_ablation/`.


In [ ]:

from pathlib import Path
import os, json, shutil, gc, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from IPython.display import display

# Keep the machine responsive.
torch.set_num_threads(min(4, os.cpu_count() or 1))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

DEVICE = torch.device("cpu")  # exact retained manuscript-facing evaluation path

SEED = 42
NUM_PHASES = 8
HIDDEN_DIM = 128
CHUNK_SIZE = 100
TAU = 0.30


def find_repo_root(start=None):
    """Locate the DMET-SSTKG repository root from the current working directory."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "Codes").exists():
            return candidate
    raise FileNotFoundError(
        "DMET-SSTKG repository root not found. Start Jupyter from the repository root "
        "or from a subdirectory inside the repository."
    )


def first_existing_file(candidates, glob_patterns=()):
    """Resolve a generated artifact without embedding machine-specific absolute paths."""
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.is_file():
            return candidate
    for pattern in glob_patterns:
        matches = sorted(REPO_ROOT.glob(pattern))
        if matches:
            return matches[-1]
    # Return the first documented candidate so the preflight error is informative.
    return Path(candidates[0])

REPO_ROOT = find_repo_root()
ARTIFACT_DIR = REPO_ROOT / "artifacts"

ROLE_NORMALIZED_STKG = (
    REPO_ROOT / "S-STKG_Data" / "SurgicalSTKG_Local_Cholec80_M2CAI"
)

CKPT_PATH = first_existing_file(
    [
        ARTIFACT_DIR / "best_sstkg_phase_classifier.pt",
        ARTIFACT_DIR / "best_robust_model.pt",
    ],
    [
        "outputs/phase_classifier_runs/*/best_sstkg_phase_classifier.pt",
        "outputs/phase_classifier_runs/*/best_robust_model.pt",
    ],
)
PHASE_RESULTS_DIR = CKPT_PATH.parent
SPLIT_PATH = PHASE_RESULTS_DIR / "data_split.npz"
META_PATH = PHASE_RESULTS_DIR / "fragment_metadata.csv"
SPLIT_CASES_PATH = PHASE_RESULTS_DIR / "split_cases.json"

AUDIT_ROOT = (
    REPO_ROOT / "outputs" / "analysis_audits" / "source_sstkg_component_ablation"
)
AUDIT_CACHE = AUDIT_ROOT / "exact_cache"
AUDIT_META = AUDIT_CACHE / "audit_fragment_metadata.csv"

# Required only for the temporal-sampling condition.
PRE_ROLE_ROOT_01_10 = (
    REPO_ROOT / "reconstruction_inputs" / "source_prerole_cases_01_10"
)
PRE_ROLE_ROOT_21_30 = (
    REPO_ROOT / "reconstruction_inputs" / "source_prerole_cases_21_30"
)

TEMPORAL_CACHE = AUDIT_ROOT / "temporal_sampling_rate10_cache"
OUTPUT_DIR = AUDIT_ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Node feature layout used by the retained classifier.
COL_X, COL_Y, COL_W, COL_H = 0, 1, 2, 3
COL_VX, COL_VY = 4, 5
COL_ROLE = 6
COL_CASE = 7
COL_G, COL_P = 8, 9
COL_ISV = 10

EXPECTED = {
    "accuracy": 82.95,
    "macro_f1": 73.88,
    "weighted_f1": 83.14,
}
BASELINE_TOL_PP = 0.20

for p in [ROLE_NORMALIZED_STKG, CKPT_PATH, SPLIT_PATH]:
    assert p.exists(), f"Required retained artifact not found: {p}"

print("Device:", DEVICE)
print("PyTorch CPU threads:", torch.get_num_threads())
print("Source S-STKG:", ROLE_NORMALIZED_STKG)
print("Phase-classifier checkpoint:", CKPT_PATH)
print("Output:", OUTPUT_DIR)



## 1. Rebuild/reuse the exact retained binary cache safely

The current historical cache path is deliberately **not used**, because it was rebuilt after the retained experiment.

The code below follows the retained source classifier's own text→binary conversion:
- cases sorted;
- phase prefixes sorted;
- frames ordered by `g_ratio`;
- chunks of 100 sampled graphs;
- final chunks shorter than 10 dropped;
- `case_id` zeroed before model input.

Only one phase file is held in memory at a time.


In [ ]:

def metadata_identity(df):
    cols = ["idx", "case_id", "phase_id", "fragment_file", "prefix"]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"fragment_metadata.csv lacks columns: {missing}")
    out = df[cols].copy()
    out["idx"] = out["idx"].astype(int)
    out["case_id"] = out["case_id"].astype(str)
    out["phase_id"] = out["phase_id"].astype(int)
    out["fragment_file"] = out["fragment_file"].astype(str)
    out["prefix"] = out["prefix"].astype(str)
    return out.sort_values("idx").reset_index(drop=True)

retained_meta = (
    metadata_identity(pd.read_csv(META_PATH))
    if META_PATH.exists()
    else None
)

def zero_based_label(raw_label):
    raw_label = int(raw_label)
    if 1 <= raw_label <= NUM_PHASES:
        return raw_label - 1
    if 0 <= raw_label < NUM_PHASES:
        return raw_label
    raise ValueError(f"Unexpected phase label: {raw_label}")

def audit_cache_matches():
    if not AUDIT_CACHE.exists() or not AUDIT_META.exists():
        return False
    try:
        m = metadata_identity(pd.read_csv(AUDIT_META))
        files = sorted(AUDIT_CACHE.glob("frag_*.pt"))
        return (retained_meta is None or retained_meta.equals(m)) and len(files) > 0
    except Exception:
        return False

def rebuild_exact_audit_cache():
    if AUDIT_CACHE.exists():
        shutil.rmtree(AUDIT_CACHE)
    AUDIT_CACHE.mkdir(parents=True, exist_ok=True)

    rows = []
    global_idx = 0
    case_dirs = sorted(
        [p for p in ROLE_NORMALIZED_STKG.iterdir()
         if p.is_dir() and p.name.startswith("Case_")],
        key=lambda p: p.name
    )

    if len(case_dirs) != 20:
        raise RuntimeError(
            f"Expected 20 source-development case directories, found {len(case_dirs)}"
        )

    for case_dir in case_dirs:
        prefixes = sorted(
            p.name.replace("_nodes.txt", "")
            for p in case_dir.glob("*_nodes.txt")
        )

        for prefix in prefixes:
            nodes_path = case_dir / f"{prefix}_nodes.txt"
            edges_path = case_dir / f"{prefix}_edges.txt"
            edge_feat_path = case_dir / f"{prefix}_edge_features.txt"
            labels_path = case_dir / f"{prefix}_labels.txt"

            if not all(p.exists() for p in
                       [nodes_path, edges_path, edge_feat_path, labels_path]):
                continue

            nodes = pd.read_csv(nodes_path)
            edges = pd.read_csv(edges_path)
            edge_feat = pd.read_csv(edge_feat_path)
            raw_label = int(pd.read_csv(labels_path).iloc[0, 1])
            label = zero_based_label(raw_label)

            unique_grs = sorted(nodes.iloc[:, COL_G].unique())

            for start in range(0, len(unique_grs), CHUNK_SIZE):
                chunk_grs = unique_grs[start:start + CHUNK_SIZE]
                if len(chunk_grs) < 10:
                    continue

                seq = []
                for gr in chunk_grs:
                    f_n = nodes[nodes.iloc[:, COL_G] == gr].values.copy()
                    f_n[:, COL_CASE] = 0.0

                    f_e = edges[edges.iloc[:, 0] == gr].values
                    f_ef = edge_feat[edge_feat.iloc[:, 0] == gr].values

                    if len(f_e):
                        edge_index = torch.tensor(
                            f_e[:, 2:4].astype(np.int64),
                            dtype=torch.long
                        ).t().contiguous()
                    else:
                        edge_index = torch.zeros((2, 0), dtype=torch.long)

                    if len(f_ef):
                        edge_attr = torch.tensor(
                            f_ef[:, 4:7],
                            dtype=torch.float
                        )
                    else:
                        edge_attr = torch.zeros((0, 3), dtype=torch.float)

                    d = Data(
                        x=torch.tensor(f_n, dtype=torch.float),
                        edge_index=edge_index,
                        edge_attr=edge_attr,
                        y=torch.tensor([label], dtype=torch.long),
                    )
                    d.is_v = d.x[:, COL_ISV] == 1
                    seq.append(d)

                fname = f"frag_{global_idx:06d}.pt"
                torch.save(
                    {
                        "graph_seq": seq,
                        "metadata": {
                            "case_id": case_dir.name,
                            "prefix": prefix,
                        },
                    },
                    AUDIT_CACHE / fname
                )

                rows.append({
                    "idx": global_idx,
                    "case_id": case_dir.name,
                    "phase_id": label,
                    "fragment_file": fname,
                    "prefix": prefix,
                })
                global_idx += 1

            del nodes, edges, edge_feat
            gc.collect()

    m = pd.DataFrame(rows)
    m.to_csv(AUDIT_META, index=False)
    return metadata_identity(m)

if audit_cache_matches():
    print("[OK] Reusing previously validated exact audit cache.")
    audit_meta = metadata_identity(pd.read_csv(AUDIT_META))
else:
    print("[INFO] Exact audit cache not available; rebuilding safely from retained S-STKG text.")
    audit_meta = rebuild_exact_audit_cache()

if retained_meta is not None and not retained_meta.equals(audit_meta):
    print(retained_meta.compare(audit_meta).head(20))
    raise RuntimeError(
        "Rebuilt fragment identity does not match the saved phase-run metadata."
    )
if retained_meta is None:
    retained_meta = audit_meta.copy()

BASE_FILES = [AUDIT_CACHE / f"frag_{i:06d}.pt" for i in range(len(retained_meta))]
assert all(p.exists() for p in BASE_FILES)

split = np.load(SPLIT_PATH)
train_idx = split["train_idx"].astype(int)
val_idx = split["val_idx"].astype(int)
test_idx = split["test_idx"].astype(int)

print("Fragments:", len(BASE_FILES))
print("Train / Val / Test:", len(train_idx), len(val_idx), len(test_idx))
assert (len(train_idx), len(val_idx), len(test_idx)) == (1275, 287, 264)

if SPLIT_CASES_PATH.exists():
    split_cases = json.loads(SPLIT_CASES_PATH.read_text())
    print("Saved test cases:", split_cases.get("test_cases"))


In [ ]:

def load_base(idx):
    obj = torch.load(BASE_FILES[int(idx)], map_location="cpu", weights_only=False)
    return obj["graph_seq"] if isinstance(obj, dict) else obj

class RobustMosaicGNN(nn.Module):
    # Exact retained source phase-classifier architecture/forward.
    def __init__(self, in_channels=11, edge_dim=3, hidden_dim=128, num_phases=8):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_dim, edge_dim=edge_dim)
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim, edge_dim=edge_dim)
        self.context_attn = nn.Linear(hidden_dim * 2, 1)
        self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 3, num_phases)

    def forward(self, graph_seq):
        frame_embs = []

        for data in graph_seq:
            x = self.conv1(data.x, data.edge_index, data.edge_attr).relu()
            x = self.conv2(x, data.edge_index, data.edge_attr).relu()

            mean_emb = x.mean(dim=0, keepdim=True)

            v_node_emb = x[data.is_v]
            if v_node_emb.size(0) == 0:
                v_node_emb = mean_emb
            else:
                v_node_emb = v_node_emb.mean(dim=0, keepdim=True)

            combined = torch.cat([mean_emb, v_node_emb], dim=-1)
            attn_weight = torch.sigmoid(self.context_attn(combined))
            v = attn_weight * v_node_emb + (1.0 - attn_weight) * mean_emb
            frame_embs.append(v)

        seq_tensor = torch.stack(frame_embs).squeeze(1).unsqueeze(0)
        out, _ = self.rnn(seq_tensor)

        global_skip = seq_tensor.mean(dim=1)
        rnn_final = out[:, -1, :]
        return self.fc(torch.cat([rnn_final, global_skip], dim=-1))

model = RobustMosaicGNN().to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model.load_state_dict(state, strict=True)
model.eval()

def metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred) * 100.0
    _, _, mf1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=range(NUM_PHASES),
        average="macro", zero_division=0
    )
    _, _, wf1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=range(NUM_PHASES),
        average="weighted", zero_division=0
    )
    return {
        "accuracy": float(acc),
        "macro_f1": float(mf1) * 100.0,
        "weighted_f1": float(wf1) * 100.0,
    }

@torch.inference_mode()
def predict_base_indices(indices, transform=None):
    yt, yp, used = [], [], []

    for idx in indices:
        seq0 = load_base(idx)
        seq = seq0 if transform is None else transform(seq0)

        if seq is None or len(seq) == 0:
            continue

        seq = [d.to(DEVICE) for d in seq]
        logits = model(seq)

        yt.append(int(seq0[0].y.item()))
        yp.append(int(logits.argmax(1).item()))
        used.append(int(idx))

        del seq, seq0, logits

    return np.asarray(yt), np.asarray(yp), np.asarray(used, dtype=int)

# Publication baseline gate.
yt, yp, used = predict_base_indices(test_idx)
baseline = metrics(yt, yp)

print("Retained held-out baseline:", baseline)
print("N:", len(used))

for key, expected in EXPECTED.items():
    if abs(baseline[key] - expected) > BASELINE_TOL_PP:
        raise RuntimeError(
            f"Baseline mismatch for {key}: got {baseline[key]:.2f}, "
            f"expected about {expected:.2f}. Stop before ablation."
        )

print("[OK] Retained manuscript-facing baseline reproduced.")



## 2. Exact one-factor-at-a-time component perturbations

Each condition alters only the named S-STKG component while the retained classifier and all remaining inputs are held unchanged.

For progress, the two progress channels are treated jointly as one temporal progress component. Their source-training means are estimated in a streaming frame-level pass and then used as fixed constants at evaluation time. This removes temporally varying progress information while avoiding the artificial interpretation that every frame occurs at progress value zero.

For `w/o virtual-context node`, a sampled frame containing no physical detections has no graph after the virtual node is removed, so that frame is omitted. If an entire fragment has no physical frame, that fragment is not evaluable for this condition. The reported delta is therefore always computed against the full S-STKG on the same evaluable fragment subset.


In [ ]:

def clone_graph(d0):
    d = d0.clone()
    d.x = d0.x.clone()
    d.edge_index = d0.edge_index.clone()
    d.edge_attr = d0.edge_attr.clone()
    d.y = d0.y.clone()
    d.is_v = d0.is_v.clone().bool()
    return d

def no_geometry(seq0):
    out = []
    for d0 in seq0:
        d = clone_graph(d0)
        d.x[:, COL_X:COL_H + 1] = 0.0
        out.append(d)
    return out

def no_motion(seq0):
    out = []
    for d0 in seq0:
        d = clone_graph(d0)
        d.x[:, COL_VX:COL_VY + 1] = 0.0
        out.append(d)
    return out

def no_role(seq0):
    out = []
    for d0 in seq0:
        d = clone_graph(d0)
        physical = ~d.is_v
        # Remove physical role distinctions only.
        # Virtual-context identity remains a separate structural component.
        d.x[physical, COL_ROLE] = 0.0
        out.append(d)
    return out

def compute_source_training_progress_means(indices):
    """
    Estimate fixed progress references from source-training frames only.

    Each frame receives equal weight. Progress values are expected to be
    frame-level quantities shared by all nodes in the same graph; the mean
    across nodes is used defensively before accumulating the frame statistic.
    """
    g_sum = 0.0
    p_sum = 0.0
    n_frames = 0

    for idx in indices:
        seq = load_base(idx)
        for d in seq:
            g_sum += float(d.x[:, COL_G].mean().item())
            p_sum += float(d.x[:, COL_P].mean().item())
            n_frames += 1
        del seq

    if n_frames == 0:
        raise RuntimeError("No source-training frames available for progress statistics.")

    return g_sum / n_frames, p_sum / n_frames, n_frames


TRAIN_MEAN_G, TRAIN_MEAN_P, TRAIN_PROGRESS_N_FRAMES = (
    compute_source_training_progress_means(train_idx)
)

print(
    "Source-training progress references "
    f"(frame-level; n={TRAIN_PROGRESS_N_FRAMES} frames): "
    f"g_ratio={TRAIN_MEAN_G:.6f}, p_ratio={TRAIN_MEAN_P:.6f}"
)


def no_progress(seq0):
    out = []
    for d0 in seq0:
        d = clone_graph(d0)
        # Neutralize temporal progress cues with source-training references.
        d.x[:, COL_G] = TRAIN_MEAN_G
        d.x[:, COL_P] = TRAIN_MEAN_P
        out.append(d)
    return out


def no_edge_descriptors(seq0):
    out = []
    for d0 in seq0:
        d = clone_graph(d0)
        d.edge_attr.zero_()
        out.append(d)
    return out

def remove_virtual_graph(d0):
    d = clone_graph(d0)
    keep = ~d.is_v

    if int(keep.sum()) == 0:
        return None

    old_to_new = torch.full((d.x.size(0),), -1, dtype=torch.long)
    old_to_new[keep] = torch.arange(int(keep.sum()), dtype=torch.long)

    if d.edge_index.numel() > 0:
        src, dst = d.edge_index
        edge_keep = keep[src] & keep[dst]
        edge_index = old_to_new[d.edge_index[:, edge_keep]]
        edge_attr = d.edge_attr[edge_keep]
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 3), dtype=torch.float)

    out = Data(
        x=d.x[keep],
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=d.y.clone(),
    )
    out.is_v = torch.zeros(out.x.size(0), dtype=torch.bool)
    return out

def no_virtual(seq0):
    out = []
    for d0 in seq0:
        d = remove_virtual_graph(d0)
        if d is not None:
            out.append(d)
    return out

NON_TEMPORAL = [
    ("Full S-STKG", None),
    ("w/o node geometry", no_geometry),
    ("w/o motion", no_motion),
    ("w/o role semantics", no_role),
    ("w/o progress", no_progress),
    ("w/o virtual-context node", no_virtual),
    ("w/o edge descriptors", no_edge_descriptors),
]



## 3. True temporal-sampling condition: source sampling rate 5 → 10

This section does **not** use a post-hoc `seq[::2]` approximation.

For the held-out source cases it reads the original **pre-role node files** generated by the development construction code. Those files retain the original detector class IDs needed for motion recomputation.

For each original phase fragment:

1. the notebook identifies the true rate-10 temporal grid as positions 0,10,20,… from the original video sampling;
2. it verifies that every corresponding pre-role frame reproduces the retained node tensor after the study's role mapping and `case_id` cleaning;
3. it uses the **exact retained graph** for each selected frame, so the study's stored topology and edge descriptors are preserved without re-implementing them;
4. it recomputes only `vx,vy` across the retained rate-10 grid using the original detector class IDs.

This is the controlled construction-level comparison required here: frame selection and the temporally dependent motion channels change with the sampling interval, while all other per-frame graph content is kept exactly as in the retained source representation.

### Why topology is inherited rather than reconstructed

The physical topology and `[dist, rel_dist, IoU]` descriptors are functions of the detections and geometry **within a selected frame**. Those detections/geometry do not change merely because the temporal sampling interval changes from 5 to 10. Reusing the retained graph for a selected frame therefore keeps those components exactly fixed and avoids introducing a second edge-construction implementation into the temporal ablation.


In [ ]:

ORIGINAL_CLASS_TO_ROLE = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5,
    6: 6,
    7: 7,
    8: 7,
    99: 8,
}

def resolve_prerole_case_dir(case_id):
    """
    Follow the ORIGINAL development construction calls deterministically:
      Case_01 ... Case_10 -> cases 01–10 reconstruction root
      Case_21 ... Case_30 -> cases 21–30 reconstruction root
    """
    case_id = str(case_id)
    try:
        case_num = int(case_id.split("_")[-1])
    except Exception as exc:
        raise RuntimeError(f"Unexpected case ID format: {case_id!r}") from exc

    if 1 <= case_num <= 10:
        chosen = PRE_ROLE_ROOT_01_10 / case_id
    elif 21 <= case_num <= 30:
        chosen = PRE_ROLE_ROOT_21_30 / case_id
    else:
        raise RuntimeError(
            f"Case {case_id} is outside retained source IDs (01-10, 21-30)."
        )

    if not chosen.exists():
        raise FileNotFoundError(
            f"Original development S-STKG directory not found for {case_id}: {chosen}"
        )
    return chosen

def resolve_prerole_node_file(case_id, prefix):
    case_dir = resolve_prerole_case_dir(case_id)

    exact = case_dir / f"{prefix}_nodes.txt"
    if exact.exists():
        return exact

    matches = [
        p for p in case_dir.glob("*_nodes.txt")
        if p.name[:-len("_nodes.txt")] == str(prefix)
    ]
    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not resolve original pre-role node file for "
        f"{case_id}, prefix={prefix!r} in {case_dir}"
    )

def expected_retained_x_from_prerole_frame(frame_df):
    """
    Convert one ORIGINAL pre-role rate-5 frame to the retained node-feature
    representation, without touching topology or edge descriptors.

    This is used only as a source-alignment gate.
    """
    f = frame_df.copy().reset_index(drop=True)

    mapped_roles = [
        ORIGINAL_CLASS_TO_ROLE[int(c)]
        for c in f["class"].astype(int).tolist()
    ]

    f["class"] = mapped_roles
    f["case_id"] = 0.0

    arr = f[
        ["x","y","w","h","vx","vy","class",
         "case_id","g_ratio","p_ratio","isV"]
    ].to_numpy(dtype=np.float32)

    return torch.tensor(arr, dtype=torch.float)

def recompute_rate10_motion(frame_df, prev_centers):
    """
    Recompute vx,vy on the rate-10 temporal grid using the ORIGINAL detector
    class IDs, exactly matching the dependency used during source construction.

    Returns an [num_nodes, 2] tensor in the original node-row order.
    """
    f = frame_df.copy().reset_index(drop=True)
    motion = np.zeros((len(f), 2), dtype=np.float32)

    for row_i, row in f.iterrows():
        if int(row["isV"]) == 1:
            motion[row_i] = (0.0, 0.0)
            continue

        cls = int(row["class"])
        x = float(row["x"])
        y = float(row["y"])

        px, py = prev_centers.get(cls, (x, y))
        vx = x - px
        vy = y - py
        prev_centers[cls] = (x, y)

        # Development node files store four decimals.
        motion[row_i, 0] = round(vx, 4)
        motion[row_i, 1] = round(vy, 4)

    return torch.tensor(motion, dtype=torch.float)

def clone_with_rate10_motion(base_graph, motion):
    """
    Keep the retained graph's EXACT nodes/topology/edge descriptors and change
    only vx,vy, which are the per-frame quantities affected by sampling interval.
    """
    d = base_graph.clone()
    d.x = base_graph.x.clone()
    d.edge_index = base_graph.edge_index.clone()
    d.edge_attr = base_graph.edge_attr.clone()
    d.y = base_graph.y.clone()
    d.is_v = base_graph.is_v.clone().bool()

    if motion.shape[0] != d.x.shape[0]:
        raise RuntimeError(
            f"Motion/node-count mismatch: motion={motion.shape[0]}, graph={d.x.shape[0]}"
        )

    d.x[:, COL_VX:COL_VY + 1] = motion
    return d


In [ ]:

# Deterministic original-root preflight for the retained held-out cases.
# This must resolve Case_03/Case_04 to the cases 01–10 root and Case_22 to the cases 21–30 root.
for _case in ["Case_03", "Case_04", "Case_22"]:
    print(_case, "->", resolve_prerole_case_dir(_case))


In [ ]:

# Build test metadata with a stable chunk ordinal inside each original
# (case, phase-fragment prefix). Do this before filtering to test.
meta_with_ord = retained_meta.sort_values("idx").copy()
meta_with_ord["chunk_ordinal"] = (
    meta_with_ord.groupby(["case_id", "prefix"], sort=False).cumcount()
)
test_meta = (
    meta_with_ord[meta_with_ord["idx"].isin(test_idx)]
    .sort_values("idx")
    .reset_index(drop=True)
)

if TEMPORAL_CACHE.exists():
    shutil.rmtree(TEMPORAL_CACHE)
TEMPORAL_CACHE.mkdir(parents=True, exist_ok=True)

temporal_files = {}
node_alignment_checked = 0

grouped = test_meta.groupby(["case_id", "prefix"], sort=False)

for (case_id, prefix), group in grouped:
    node_path = resolve_prerole_node_file(case_id, prefix)
    raw_nodes = pd.read_csv(node_path)
    raw_nodes.columns = [c.strip() for c in raw_nodes.columns]

    required_cols = {
        "x","y","w","h","vx","vy","class","case_id","g_ratio","p_ratio","isV"
    }
    missing = sorted(required_cols.difference(raw_nodes.columns))
    if missing:
        raise RuntimeError(f"{node_path} missing columns: {missing}")

    unique_grs = sorted(raw_nodes["g_ratio"].unique())

    # True global rate-10 grid within this original phase fragment:
    # original rate-5 sampled positions 0,2,4,... correspond to raw 0,10,20,...
    selected_rate10_grs = set(unique_grs[::2])

    # Recompute temporal motion ONCE across the complete selected rate-10
    # sequence for this original phase fragment, so chunk boundaries do not
    # reset motion state.
    prev_centers_rate10 = {}
    rate10_motion_by_gr = {}

    for gr in unique_grs:
        if gr not in selected_rate10_grs:
            continue

        frame_df = (
            raw_nodes[raw_nodes["g_ratio"] == gr]
            .reset_index(drop=True)
        )

        rate10_motion_by_gr[gr] = recompute_rate10_motion(
            frame_df,
            prev_centers_rate10,
        )

    for row in group.itertuples(index=False):
        idx = int(row.idx)
        chunk_ord = int(row.chunk_ordinal)

        start = chunk_ord * CHUNK_SIZE
        base_chunk_grs = unique_grs[start:start + CHUNK_SIZE]

        base_seq = load_base(idx)

        if len(base_chunk_grs) != len(base_seq):
            raise RuntimeError(
                f"Original-source alignment length mismatch at idx={idx}: "
                f"{len(base_chunk_grs)} source frames vs {len(base_seq)} retained graphs"
            )

        seq10 = []

        for pos, gr in enumerate(base_chunk_grs):
            frame_df = (
                raw_nodes[raw_nodes["g_ratio"] == gr]
                .reset_index(drop=True)
            )

            # Strong source-alignment gate:
            # the original pre-role node frame must reproduce the retained
            # node tensor after the study's role mapping + case_id cleaning.
            expected_x = expected_retained_x_from_prerole_frame(frame_df)
            retained_graph = base_seq[pos]

            if expected_x.shape != retained_graph.x.shape:
                raise RuntimeError(
                    f"Node-shape mismatch at idx={idx}, pos={pos}, g_ratio={gr}: "
                    f"{expected_x.shape} vs {retained_graph.x.shape}"
                )

            if not torch.allclose(
                expected_x,
                retained_graph.x.cpu(),
                atol=1.1e-4,
                rtol=0,
            ):
                max_diff = float(
                    (expected_x - retained_graph.x.cpu()).abs().max().item()
                )
                raise RuntimeError(
                    f"Original pre-role -> retained node gate failed at "
                    f"idx={idx}, frame_position={pos}, g_ratio={gr}; "
                    f"max node-feature diff={max_diff}"
                )

            node_alignment_checked += 1

            # For the rate-10 condition, keep this exact retained frame graph
            # only when it belongs to the true 0,10,20,... grid. Topology and
            # edge descriptors are unchanged because they are per-frame
            # functions of the same detections/geometry. Only motion must be
            # recomputed because it depends on the temporal sampling interval.
            if gr in selected_rate10_grs:
                seq10.append(
                    clone_with_rate10_motion(
                        retained_graph,
                        rate10_motion_by_gr[gr],
                    )
                )

        if len(seq10) == 0:
            raise RuntimeError(
                f"Rate-10 produced an empty sequence for retained idx={idx}"
            )

        out_file = TEMPORAL_CACHE / f"frag_{idx:06d}.pt"
        torch.save(
            {
                "graph_seq": seq10,
                "metadata": {
                    "retained_idx": idx,
                    "case_id": str(case_id),
                    "prefix": str(prefix),
                    "sampling_rate": 10,
                    "topology_source": "exact_retained_frame_graph",
                    "motion_source": "recomputed_original_detector_class_id",
                },
            },
            out_file,
        )
        temporal_files[idx] = out_file

    del raw_nodes, rate10_motion_by_gr
    gc.collect()

assert set(temporal_files) == set(map(int, test_idx))

print("[OK] Original pre-role -> retained NODE alignment gate passed.")
print("Retained frame-node tensors checked:", node_alignment_checked)
print("Rate-10 test fragments created:", len(temporal_files))


In [ ]:

def load_temporal10(idx):
    obj = torch.load(temporal_files[int(idx)], map_location="cpu", weights_only=False)
    return obj["graph_seq"]

@torch.inference_mode()
def predict_temporal10(indices):
    yt, yp, used = [], [], []

    for idx in indices:
        seq = load_temporal10(idx)
        seq = [d.to(DEVICE) for d in seq]

        logits = model(seq)
        yt.append(int(seq[0].y.item()))
        yp.append(int(logits.argmax(1).item()))
        used.append(int(idx))

        del seq, logits

    return np.asarray(yt), np.asarray(yp), np.asarray(used, dtype=int)



## 4. Run the final component-ablation experiment

All conditions use the same retained classifier and held-out source-test cases. The progress condition is a single unified ablation in which temporally varying progress cues are replaced by fixed source-training references.

`Delta Weighted-F1` is computed against the full S-STKG on the same evaluable fragment subset. For all conditions except possible `w/o virtual-context node`, that subset is the complete 264-fragment source-test split.


In [ ]:

rows = []

# Non-temporal components.
for label, transform in NON_TEMPORAL:
    print(f"Running: {label}")

    yt_c, yp_c, used_c = predict_base_indices(test_idx, transform=transform)
    m_c = metrics(yt_c, yp_c)

    # Matched full reference for exactly the same fragment subset.
    yt_r, yp_r, used_r = predict_base_indices(used_c, transform=None)
    assert np.array_equal(used_c, used_r)
    m_r = metrics(yt_r, yp_r)

    rows.append({
        "S-STKG condition": label,
        "N": len(used_c),
        "Accuracy (%)": m_c["accuracy"],
        "Macro-F1 (%)": m_c["macro_f1"],
        "Weighted-F1 (%)": m_c["weighted_f1"],
        "Matched full Weighted-F1 (%)": m_r["weighted_f1"],
        "Delta Weighted-F1 (pp)": m_c["weighted_f1"] - m_r["weighted_f1"],
    })

# True temporal sampling condition.
print("Running: temporal sampling rate 10 (vs 5)")
yt_t, yp_t, used_t = predict_temporal10(test_idx)
m_t = metrics(yt_t, yp_t)

yt_r, yp_r, used_r = predict_base_indices(used_t, transform=None)
assert np.array_equal(used_t, used_r)
m_r = metrics(yt_r, yp_r)

rows.append({
    "S-STKG condition": "temporal sampling rate 10 (vs 5)",
    "N": len(used_t),
    "Accuracy (%)": m_t["accuracy"],
    "Macro-F1 (%)": m_t["macro_f1"],
    "Weighted-F1 (%)": m_t["weighted_f1"],
    "Matched full Weighted-F1 (%)": m_r["weighted_f1"],
    "Delta Weighted-F1 (pp)": m_t["weighted_f1"] - m_r["weighted_f1"],
})

results = pd.DataFrame(rows)

# Keep the requested publication table compact.
publication = results[[
    "S-STKG condition",
    "N",
    "Accuracy (%)",
    "Weighted-F1 (%)",
    "Delta Weighted-F1 (pp)",
]].copy()

results.to_csv(
    OUTPUT_DIR / "sstkg_component_ablation_full_diagnostics.csv",
    index=False
)
publication.to_csv(
    OUTPUT_DIR / "sstkg_component_ablation_table.csv",
    index=False
)

print("\nFINAL TABLE")
display(publication.round(2))

print("\nFull diagnostics")
display(results.round(2))

print("\nSaved:")
print(OUTPUT_DIR / "sstkg_component_ablation_table.csv")
print(OUTPUT_DIR / "sstkg_component_ablation_full_diagnostics.csv")



## Reporting language

This experiment is a targeted fixed-checkpoint S-STKG component ablation on the held-out source-test split.

A concise Methods description is:

> To assess representation-level dependence on specific S-STKG construction components, the retained phase-classifier checkpoint was kept fixed while one component was altered at a time on the held-out source-test split. Progress information was neutralized by replacing the progress channels with fixed source-training means, thereby removing temporal progress cues while preserving their input-scale reference. Temporal sampling was evaluated separately by increasing the source construction interval from 5 to 10 frames, with motion recomputed on the corresponding sparser frame grid.

Interpret the progress row as sensitivity to temporally varying progress cues under the retained classifier, not as an independently retrained model without progress features.


## Manuscript-facing expected result

The complete run reproduces **Table 4** of the manuscript. A compact machine-readable reference is included in `expected_results/table4_sstkg_component_ablation.csv`.

The temporal-sampling row requires the reconstruction-route pre-role inputs described above; the remaining component perturbations use the released source S-STKG representation and the retained source classifier.
